In [2]:
import sqlite3 as sq
import datetime

In [4]:
class Store:
  def __init__(self, db_name="inventory.db"):
    self.conn = sq.connect(db_name)
    self.cursor = self.conn.cursor()
    self.create_database()

  def create_database(self):
    statement_create_products_table = '''CREATE TABLE IF NOT EXISTS Product(
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    product_price REAL NOT NULL,
    product_quantity INTEGER NOT NULL 
    )'''

    statement_create_sales_table = '''CREATE TABLE IF NOT EXISTS Sales(
    sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
    sale_date TEXT NOT NULL,
    product_name TEXT NOT NULL,
    sale_total REAL NOT NULL
    )'''

    self.cursor.execute(statement_create_products_table)
    self.cursor.execute(statement_create_sales_table)
    self.conn.commit()

  def addProduct(self, name, price, quantity):
    statement = '''INSERT INTO Product (product_name, product_price, product_quantity) VALUES (?, ?, ?)'''
    params = (name, price, quantity)
    if price <= 0 or quantity < 0:
      raise ValueError("Price must be greater than 0 and quantity must be non-negative")
    self.cursor.execute(statement, params)
    self.conn.commit()

  def removeProduct(self, product_id):
    statement = '''DELETE FROM Product WHERE product_id = ?'''
    param = (product_id,)
    self.cursor.execute(statement, param)
    self.conn.commit()
    return self.cursor.rowcount > 0

  def updateProduct(self,product_id, name=None, price=None, quantity=None):
    updates = []
    params = []
    if name is not None:
      updates.append("product_name = ?")
      params.append(name)
    
    if price is not None:
      if price <= 0:
        raise ValueError("Price must be positive")
      updates.append("product_price = ?")
      params.append(price)
    
    if quantity is not None:
      if quantity < 0:
        raise ValueError("Quantity cannot be negative")
      updates.append("product_quantity = ?")
      params.append(quantity)
    if not updates:
      return False
    
    params.append(product_id)
    query = f"UPDATE Product SET {', '.join(updates)} WHERE product_id = ?"
    self.cursor.execute(query, params)
    self.conn.commit()
    return self.cursor.rowcount > 0

  def displayProducts(self):
    self.cursor.execute("SELECT * FROM Product")
    return self.cursor.fetchall()

  def sellProduct(self, product_id, quantity):
    if quantity <= 0:
      raise ValueError("Quantity must be positive")
    statement = '''SELECT product_name, product_price, product_quantity FROM Product WHERE product_id = ?'''
    self.cursor.execute(statement, (product_id,))
    product = self.cursor.fetchone()
    if not product:
      return False, "Product not found."
    name, price, cur_quantity = product
    if cur_quantity < quantity:
      return False, "Not enough quantity"
    new_quantity = cur_quantity - quantity
    try:
      update_statement = '''UPDATE Product SET product_quantity = ? WHERE product_id = ?'''
      data = (new_quantity, product_id)
      self.cursor.execute(update_statement, data)
      sale_date = datetime.date.today().isoformat()
      total = price * quantity

      insert_statement = '''INSERT INTO Sales (sale_date, product_name, sale_total) VALUES (?, ?, ?)'''
      insert_parame = (sale_date, name, total)
      self.cursor.execute(insert_statement, insert_parame)

      self.conn.commit()
      return True, "Sale completed successfully."
    except Exception as e:
      self.conn.rollback()
      print(f"Error during sale: {e}") 
      return False, f"Error during sale {e}"

In [6]:
def get_choice():
    try:
      choice = int(input("Select an option (1-6): "))
      if choice < 1 or choice > 6:
        print("Error: Please choose a number between 1 and 6.")
        return get_choice()
    except ValueError:
      print("Error: Invalid input. Please try again")
      return get_choice()
    return choice

In [8]:
def main():
  store = Store()
  while True:
    print("\nPL-Kits Inventory Management\n1. Add a Product\n2. Remove a Product\n3. Update a Product\n4. Display all Products\n5. Sell a Product\n6. Exit\n\n")
    choice = get_choice()
    if choice == 1:
      try:
        name = input("Enter product name: ")
        if not name:
          print("Product name cannot be empty.")
          continue
        price = float(input("Enter product price: "))
        quantity = int(input("Enter product quantity: "))
        
      except ValueError as e:
        print(f"Invalid input: {e}")
      
      store.addProduct(name, price, quantity)
      print("Product added successfully.")

    elif choice == 2:
      try:
        product_id = int(input("Enter product ID to remove: "))
        success = store.removeProduct(product_id)
        if success:
          print("Product removed successfully.")
        else:
          print("Product not found.")
      except ValueError:
        print("Invalid input. Please enter a valid product ID.")

    elif choice == 3:
      try:
        product_id = int(input("Enter product ID to update: "))
        name = input("Enter new name (leave blank to keep current): ").strip()
        name = name if name else None
        price_str = input("Enter new price (leave blank to keep current): ").strip()
        price = float(price_str) if price_str else None
        quantity_str = input("Enter new quantity (leave blank to keep current): ").strip()
        quantity = int(quantity_str) if quantity_str else None
        updated = store.updateProduct(product_id, name, price, quantity)
        if updated:
          print("Product updated successfully.")
        else:
          print("Product not found or no changes made.")
      except ValueError as e:
        print(f"Invalid input: {e}")

    elif choice == 4:
      products = store.displayProducts()
      if not products:
        print("No products in inventory.")
      else:
        print("\nProduct List:")
        print("{:<5} {:<20} {:<10} {:<10}".format("ID", "Name", "Price", "Quantity"))
        for p in products:
          print("{:<5} {:<20} {:<10.2f} {:<10}".format(p[0], p[1], p[2], p[3]))
    
    elif choice == 5:
      try:
        product_id = int(input("Enter product ID to sell: "))
        quantity = int(input("Enter quantity to sell: "))
        success, message = store.sellProduct(product_id, quantity)
        print(message)
      except ValueError as e:
        print(f"Invalid input: {e}")
    
    elif choice == 6:
      print("Exiting application")
      break

  store.conn.close()

In [12]:
main()


PL-Kits Inventory Management
1. Add a Product
2. Remove a Product
3. Update a Product
4. Display all Products
5. Sell a Product
6. Exit




Select an option (1-6):  4



Product List:
ID    Name                 Price      Quantity  
1     Kitkat               15.00      2800      

PL-Kits Inventory Management
1. Add a Product
2. Remove a Product
3. Update a Product
4. Display all Products
5. Sell a Product
6. Exit




Select an option (1-6):  6


Exiting application
